Install & Imports

In [ ]:
!pip install timm

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import timm
import os

Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/genai_project"

train_dir = f"{BASE_DIR}/data/split/train"
val_dir   = f"{BASE_DIR}/data/split/val"
test_dir  = f"{BASE_DIR}/data/split/test"
real_dir  = f"{BASE_DIR}/data/real_test"

Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

Datasets

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=test_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=test_transform)
real_dataset  = datasets.ImageFolder(real_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)
real_loader  = DataLoader(real_dataset, batch_size=32)

print("Classes:", train_dataset.classes)
num_classes = len(train_dataset.classes)

Load DINO Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model(
    "vit_small_patch16_224.dino",
    pretrained=True,
    num_classes=0   # ❗ רק features
)

model = model.to(device)

Classifier Head

In [ ]:
classifier = nn.Linear(model.num_features, num_classes).to(device)

Freeze DINO

In [ ]:
for param in model.parameters():
    param.requires_grad = False

Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

Train Functions

In [ ]:
def train_epoch():
    model.eval()
    classifier.train()

    total_loss = 0
    correct = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            features = model(x)

        outputs = classifier(features)

        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y).sum().item()

    return total_loss / len(train_loader), correct / len(train_dataset)

Eval

In [ ]:
def eval_model(loader):
    model.eval()
    classifier.eval()

    correct = 0
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            features = model(x)
            outputs = classifier(features)

            loss = criterion(outputs, y)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

Training Loop

In [ ]:
epochs = 15

for epoch in range(epochs):

    train_loss, train_acc = train_epoch()
    val_loss, val_acc = eval_model(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f}")
    print(f"Val Loss:   {val_loss:.3f} | Val Acc:   {val_acc:.3f}")

Test

In [ ]:
syn_loss, syn_acc = eval_model(test_loader)
real_loss, real_acc = eval_model(real_loader)

print("\nSynthetic Test:", syn_acc)
print("Real Test:", real_acc)